In [2]:
import csv
import re
import shutil
from pathlib import Path
from urllib.request import Request, urlopen

project_root = Path.cwd()
if not (project_root / "books.csv").is_file():
    project_root = project_root.parents[1]
csv_path = project_root / "books.csv"
books_dir = project_root / "books"
books_dir.mkdir(parents=True, exist_ok=True)

def download_pdf(url, destination):
    temporary = destination.with_suffix(".pdf.part")
    try:
        request = Request(url, headers={"User-Agent": "BooksDownloader/1.0"})
        with urlopen(request, timeout=60) as response, temporary.open("wb") as output:
            header = response.read(1024)
            if b"%PDF-" not in header:
                raise ValueError("La resposta no és un PDF")
            output.write(header)
            shutil.copyfileobj(response, output)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)

with csv_path.open(newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    first_column = reader.fieldnames[0]
    for row in reader:
        url = (row.get("pdf_url") or "").strip()
        if not url:
            continue
        book_title = (row.get(first_column) or "").strip()
        name = re.sub(r"[^\w.-]+", "_", book_title).strip("._") or "book"
        destination = books_dir / f"{name}.pdf"
        if destination.is_file() and destination.stat().st_size > 0:
            print(f"Ja existeix: {destination.name}")
            continue
        download_pdf(url, destination)
        print(f"Descarregat: {destination.name}")

Ja existeix: Think_Python_2e.pdf
Ja existeix: Think_DSP.pdf
Ja existeix: Think_Complexity_2e.pdf
Ja existeix: Think_Java_2e.pdf
Ja existeix: Physical_Modeling_in_MATLAB.pdf
Ja existeix: Think_OS.pdf
Ja existeix: Think_C.pdf


In [3]:
from pathlib import Path
from markitdown import MarkItDown

books_dir = project_root / "books"
books_text_dir = project_root / "books_text"
books_text_dir.mkdir(parents=True, exist_ok=True)
converter = MarkItDown()
markdown_lines = {}

for pdf_path in sorted(books_dir.glob("*.pdf")):
    markdown_path = books_text_dir / f"{pdf_path.stem}.md"
    if markdown_path.is_file() and markdown_path.stat().st_size > 0:
        print(f"Ja existeix: {markdown_path.name}")
        continue
    result = converter.convert(str(pdf_path))
    markdown_path.write_text(result.text_content, encoding="utf-8")
    markdown_lines[markdown_path.name] = len(result.text_content.splitlines())
    print(f"Convertit: {markdown_path.name}")

2026-10-02 07:50:30.075114114 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Convertit: Physical_Modeling_in_MATLAB.md
Convertit: Think_C.md
Convertit: Think_Complexity_2e.md
Convertit: Think_DSP.md
Convertit: Think_Java_2e.md
Convertit: Think_OS.md
Convertit: Think_Python_2e.md


In [4]:
import json
from gitsource import chunk_documents

books_text_dir = project_root / "books_text"
chunks_path = project_root / "books_chunks" / "chunks.jsonl"
chunks_path.parent.mkdir(parents=True, exist_ok=True)

documents = []
for markdown_path in sorted(books_text_dir.glob("*.md")):
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    documents.append({
        "source": markdown_path.name,
        "content": [line for line in lines if line.strip()],
    })
# print (f"Documents: {(documents)}")

chunks = chunk_documents(documents, size=100, step=50)

with chunks_path.open("w", encoding="utf-8") as file:
    for chunk in chunks:
        file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")
print(f"Desat a: {chunks_path}")

Documents: 7
Chunks: 988
Desat a: /workspaces/ai-engineering-buildcamp-codespaces/Week1/books_chunks/chunks.jsonl


In [5]:
from minsearch import Index
import json
from pathlib import Path

from minsearch import Index


def load_chunks(path: str | Path) -> list[dict]:
    path = Path(path)

    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def prepare_documents(chunks: list[dict]) -> list[dict]:
    documents = []

    for chunk in chunks:
        document = dict(chunk)

        # content és una llista de línies al JSONL
        if isinstance(document["content"], list):
            document["content"] = "\n".join(document["content"])

        documents.append(document)

    return documents


chunks = load_chunks("books_chunks/chunks.jsonl")
documents = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(documents)

print(f"Documents indexats: {len(documents)}")
results = index.search("python function definition", num_results=5)
for i in results:
    print (i['source'])


Documents indexats: 988
Physical_Modeling_in_MATLAB.md
Think_DSP.md
Think_Python_2e.md
Physical_Modeling_in_MATLAB.md
Physical_Modeling_in_MATLAB.md


In [6]:
from openai import OpenAI
openai_client = OpenAI()

import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    response = llm(prompt, instructions)
    return response
unstructured_output = rag("python function definition")
answer = unstructured_output.output_text

print (f"Usage unstructured output: {(unstructured_output.usage)}")
print (f"Unstructured output {(answer)}")



Usage unstructured output: ResponseUsage(input_tokens=7929, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=218, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=8147)
Unstructured output To define a function in Python, you use the `def` keyword followed by the function name and parentheses containing any parameters. You then write the code for what the function should do and indicate the return value, if any.

Here’s a basic structure for defining a function:

```python
def function_name(parameters):
    # Code block
    return result  # Optional
```

### Example:

```python
def add_numbers(a, b):
    return a + b
```

In this example:
- `add_numbers` is the function name.
- `a` and `b` are parameters that the function takes as input.
- The function returns the sum of `a` and `b`.

### Calling the Function:

To call the function, you simply use its name followed by parentheses with arguments:

```python
result = 

In [7]:
from pydantic import BaseModel, Field
from typing import Literal

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    """ Aquest cop volem resposta estructurada"""

    class RAGResponse(BaseModel):
        answer: str = Field(description="The main answer to the user's question in markdown")
        found_answer: bool = Field(description="True if relevant information was found in the documentation")
        confidence: float = Field(description="Confidence score from 0.0 to 1.0")
        confidence_explanation: str = Field(description="Explanation about the confidence level")
        answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
        followup_questions: list[str] = Field(description="Suggested follow-up questions")

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return response

structured_output = rag("python function definition")
answer = structured_output.output[0].content[0].parsed
print (f"Usage structured output: {(structured_output.usage)}")
print (f"Structured output {(answer)}")


Usage structured output: ResponseUsage(input_tokens=8115, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=328, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=8443)
Structured output answer='### Python Function Definition\n\nA function in Python is defined using the `def` keyword followed by the function name and parentheses containing any parameters. The syntax is as follows:\n\n```python\ndef function_name(parameters):\n    # function body\n    return output\n```\n\n#### Example:\nHere’s a simple example of a function that adds two numbers:\n\n```python\ndef add_numbers(a, b):\n    result = a + b\n    return result\n```\n\nIn this example:\n- `add_numbers` is the name of the function.\n- It takes two parameters, `a` and `b`.\n- The function body computes the sum of `a` and `b`, and then returns the result.\n\n#### Calling a Function:\nYou can call the function like this:\n\n```python\nsum = add_numbers(3, 5)\np

In [12]:



print (f"Resposta1:  {markdown_lines['Think_Python_2e.md']} lines")


source_name = "Think_Python_2e.md"

num_chunks = sum(
    1
    for chunk in chunks
    if chunk["source"] == source_name
)

print (f"Resposta2:  {source_name}: {num_chunks} chunks")
print (f"Resposta3:  {(len(index.docs))} chunks")
print (f"Resposta4:  {results[0]['source']}")
print (f"Resposta5:  {unstructured_output.usage.input_tokens} input tokens")
print (f"Resposta6:  {structured_output.usage.input_tokens - unstructured_output.usage.input_tokens} tokens more then unstructured")

Resposta1:  9703 lines
Resposta2:  Think_Python_2e.md: 189 chunks
Resposta3:  988 chunks
Resposta4:  Physical_Modeling_in_MATLAB.md
Resposta5:  7929 input tokens
Resposta6:  186 tokens more then unstructured
